In [1]:
# --- Standard Libraries ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import shutil
import datetime
from typing import Any
import json
import time
import requests

# --- Azure Libraries ---
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment, Model
from azure.ai.ml.constants import AssetTypes
from azure.core.exceptions import ServiceResponseError

# --- ML Core & Models ---
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from xgboost import XGBClassifier, plot_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import precision_recall_curve, average_precision_score, recall_score, precision_score, f1_score, PrecisionRecallDisplay, confusion_matrix

# --- Imbalanced Handling ---
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# --- Experiment Tracking ---
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (2.7.1) and child packages mlflow-skinny (3.9.0), mlflow-tracing (3.8.1) are different. This may lead to unexpected behavior. Please install the same version of all MLflow packages.
  mlflow.mismatch._check_version_mismatch()


KeyboardInterrupt: 

In [ ]:
ml_client = MLClient.from_config(credential=DefaultAzureCredential())
print(f"Connected to Workspace: {ml_client.workspace_name}")

In [ ]:
# Helper function to make loading faster
def load_azure_csv(asset_name):
    asset = ml_client.data.get(name=asset_name, version="1")
    return pd.read_csv(asset.path)

# Load Features
X_train_imb = load_azure_csv("X-train-imb")
X_train_smote = load_azure_csv("X-train-smote")
X_test = load_azure_csv("X-test-final")

# Load Labels (and flatten to 1D array)
y_train_imb = load_azure_csv("y-train-imb").values.ravel()
y_train_smote = load_azure_csv("y-train-smote").values.ravel()
y_test = load_azure_csv("y-test-final").values.ravel()

print("All cloud assets loaded successfully!")

In [ ]:
# Point MLflow to Azure ML Workspace
mlflow.set_tracking_uri(ml_client.workspaces.get(ml_client.workspace_name).mlflow_tracking_uri)

# Set the Experiment Name
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M")
experiment_name = f"fraud-detection-tournament-{timestamp}"
mlflow.set_experiment(experiment_name)

In [ ]:
# DEFINE SCENARIOS (Data Strategies) ---
scenarios = {
    "Imbalanced": (X_train_imb, y_train_imb),
    "SMOTE": (X_train_smote, y_train_smote)
}

# Calculate the scale_pos_weight for XGBoost based on your 0.17% rate
# 284,315 / 492 ≈ 577.8
ratio = 578

# DEFINE MODELS ---
models = {
    "Random_Forest": RandomForestClassifier(
        n_estimators=100, 
        max_depth=10, 
        class_weight='balanced',
        random_state=42, 
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        scale_pos_weight=ratio,
        eval_metric='aucpr',
        random_state=42
    ),
    "Logistic_Regression": LogisticRegression(
        max_iter=1000, 
        class_weight='balanced', 
        solver='liblinear', 
        random_state=42
    )
}

In [ ]:
mlflow.sklearn.autolog(disable=True)

for scenario_name, (train_x, train_y) in scenarios.items():
    for model_name, model in models.items():
        
        with mlflow.start_run(run_name=f"{model_name}-{scenario_name}"):
            
            print(f"Training {model_name} on {scenario_name}...", flush=True)
            
            # --- TRAIN ---
            model.fit(train_x, train_y)
            
            # --- PREDICT ---
            probs = model.predict_proba(X_test)[:, 1]
            preds = model.predict(X_test)
            
            # --- CALCULATE METRICS ---
            auprc = average_precision_score(y_test, probs)
            recall = recall_score(y_test, preds)
            precision = precision_score(y_test, preds)
            f1 = f1_score(y_test, preds)
            
            # --- LOG PARAMETERS ---
            mlflow.log_param("Algorithm", model_name)
            mlflow.log_param("Data Strategy", scenario_name)
            
            # --- LOG METRICS ---
            mlflow.log_metric("AUPRC", auprc)
            mlflow.log_metric("Recall", recall)
            mlflow.log_metric("Precision", precision)
            mlflow.log_metric("F1-Score", f1)
            
            print(f"Metrics logged: AUPRC={auprc:.4f}, Recall={recall:.4f}", flush=True)

print("\n--- Tournament Complete. Check the 'Jobs' tab in Azure ML Studio to compare results. ---")

In [ ]:
# Get the Experiment ID by name
experiment = mlflow.get_experiment_by_name(experiment_name)

# Search for all runs in this experiment
# This returns a pandas DataFrame automatically!
runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

# Calculate training duration for each training
runs_df['duration_sec'] = (runs_df['end_time'] - runs_df['start_time']).dt.total_seconds()

# Clean up the view (Optional)
# MLflow adds many columns (start time, tags, etc.); let's pick the important ones
cols_to_keep = [
    'run_id',
    'params.Algorithm', 
    'params.Data Strategy',
    'duration_sec', 
    'metrics.AUPRC', 
    'metrics.Recall', 
    'metrics.Precision', 
    'metrics.F1-Score'
]

summary_df = runs_df[cols_to_keep].copy()

# Rename columns for a cleaner look
summary_df.columns = ['run_id', 'Algorithm', 'Data Strategy', 'Duration (s)', 'AUPRC', 'Recall', 'Precision', 'F1-Score']


# Rank by AUPRC descending
ranked_models = summary_df.dropna(subset=['Algorithm']).drop_duplicates(subset=['Algorithm', 'Data Strategy']).sort_values(by='AUPRC', ascending=False).reset_index(drop=True)

ranked_models = (summary_df
                 .dropna(subset=['Algorithm', 'Data Strategy'])
                 .drop_duplicates(subset=['Algorithm', 'Data Strategy'], keep='first')
                 .sort_values(by='AUPRC', ascending=False)
                 .reset_index(drop=True))

# Display the final leaderboard
display(ranked_models.drop(columns=['run_id']))

In [ ]:
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M")
XGBoost_tuning_experiment_name = f"XGBoost_hyperparameter_tuning-{timestamp}"

mlflow.set_experiment(XGBoost_tuning_experiment_name)
mlflow.sklearn.autolog(max_tuning_runs=15, log_models=True)

# 1. Define the XGBoost Pipeline
xgb_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('xgb', XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'))
])

# 2. Regularization-focused Search Space
# 'reg_alpha' and 'reg_lambda' are the "magic" parameters that stop overfitting
param_dist_xgb = {
    'xgb__n_estimators': [100, 200],
    'xgb__learning_rate': [0.01, 0.1],
    'xgb__max_depth': [3, 6],
    'xgb__reg_alpha': [0, 0.1, 1], 
    'xgb__reg_lambda': [1, 5]
}

random_search_xgb = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_dist_xgb,
    n_iter=10,
    cv=3,
    scoring='average_precision',
    n_jobs=-1,
    verbose=3
)

print("Tuning XGBoost...")
random_search_xgb.fit(X_train_imb, y_train_imb)

print(f"Best AUPRC Found: {random_search_xgb.best_score_:.4f}")
print(f"Best Params: {random_search_xgb.best_params_}")

In [ ]:
# Extract the best model from the search
best_xgb_model = random_search_xgb.best_estimator_

# Generate final probabilities on the Test Set
final_probs = best_xgb_model.predict_proba(X_test)[:, 1]

# Calculate for Tuned (Best) Model
p_tuned, r_tuned, _ = precision_recall_curve(y_test, final_probs)
ap_tuned = average_precision_score(y_test, final_probs)

# Calculate for Baseline
# p_base, r_base, _ = precision_recall_curve(y_test, baseline_probs)
# ap_base = average_precision_score(y_test, baseline_probs)

plt.figure(figsize=(10, 7))

# Plot Tuned
plt.plot(r_tuned, p_tuned, label=f'Tuned XGBoost (AP: {ap_tuned:.3f})', color='darkblue', lw=3)

# Plot Baseline (Assuming you saved baseline_probs earlier)
plt.plot(r_base, p_base, label=f'Baseline XGBoost (AP: {ap_base:.3f})', color='gray', linestyle='--')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Impact of Hyperparameter Optimization on Model Performance')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

# Plot the Original Scenarios (Imbalanced vs SMOTE Baseline)
for scenario_name, (train_x, train_y) in scenarios.items():
    # We use a basic model here to show the 'Out of the Box' performance
    model = XGBClassifier(random_state=42).fit(train_x, train_y)
    PrecisionRecallDisplay.from_estimator(
        model, X_test, y_test, ax=ax, name=f"Baseline: {scenario_name}", alpha=0.5
    )

# Add the "Tuned Champion" (Your RandomizedSearch result)
# We plot this with a thicker line and a distinct color to make it pop
PrecisionRecallDisplay.from_estimator(
    best_xgb_model, 
    X_test, 
    y_test, 
    ax=ax, 
    name="Tuned XGBoost (SMOTE + Opt)", 
    color='darkblue', 
    linewidth=3
)

plt.title("Precision-Recall Evolution: From Baseline to Tuned Model")
plt.grid(alpha=0.3)
plt.legend(loc='lower left')
plt.show()

In [ ]:
# Experiment with different policy threshold
def get_policy_metrics(df, threshold):
    """
    Finds the exact metrics in the policy_df closest to a specific threshold.
    """
    # Find the row where the threshold is closest to our target
    idx = (df['Threshold'] - threshold).abs().idxmin()
    return df.iloc[idx]

def get_target_recall_policy(df, target_recall):
    """
    Finds the threshold that hits a Recall target with max Precision.
    """
    mask = df['Recall'] >= target_recall
    return df[mask].sort_values(by='Precision', ascending=False).iloc[0]

# 1. Define Business Policies + The Baseline
target_policies = {
    "Aggressive (Security First)": 0.92,
    "Balanced (Champion)": 0.85,
    "Conservative (Low Friction)": 0.75
}

policy_results = []

# --- A. Calculate the Baseline (0.5) ---
baseline = get_policy_metrics(policy_df, 0.5)
policy_results.append({
    "Policy Mode": "Standard Baseline",
    "Threshold": 0.5000,
    "Recall": f"{baseline['Recall']*100:.1f}%",
    "Precision": f"{baseline['Precision']*100:.1f}%",
})

# --- B. Calculate Tuned Policies ---
for name, target in target_policies.items():
    match = get_target_recall_policy(policy_df, target)
    policy_results.append({
        "Policy Mode": name,
        "Threshold": round(match['Threshold'], 4),
        "Recall": f"{match['Recall']*100:.1f}%",
        "Precision": f"{match['Precision']*100:.1f}%"
    })

# 2. Display the Comparison Table
summary_table = pd.DataFrame(policy_results)
display(summary_table)

In [ ]:
# Get the Precision, Recall, and Thresholds from the Tuned Model
precision, recall, thresholds = precision_recall_curve(y_test, final_probs)

# Define the specific "Policy Points" we want to test
# We'll look at the standard 0.5 and several high-precision candidates
policy_thresholds = [0.5, 0.8, 0.9, 0.95, 0.99, 0.996]
policy_results = []

for t in policy_thresholds:
    # Apply the threshold to the probabilities
    y_pred_t = (final_probs >= t).astype(int)
    
    # Calculate Metrics
    tp = confusion_matrix(y_test, y_pred_t)[1, 1]
    fp = confusion_matrix(y_test, y_pred_t)[0, 1]
    fn = confusion_matrix(y_test, y_pred_t)[1, 0]
    
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    policy_results.append({
        "Threshold": t,
        "Precision": f"{prec:.2%}",
        "Recall": f"{rec:.2%}",
        "False Positives": fp,
        "Fraud Caught": tp
    })

# 3. Display the Policy Table
df_policy = pd.DataFrame(policy_results)
display(df_policy)

In [ ]:
# Plotting the top 10 features
fig, ax = plt.subplots(figsize=(10, 8))
plot_importance(best_xgb_model, max_num_features=10, ax=ax, importance_type='gain')
plt.title("Tuned XGBoost Feature Importance (Inference Drive)")
plt.show()

In [ ]:
with mlflow.start_run(run_name="Isolation_Forest_Training"):
    
    # Enable autologging for scikit-learn
    mlflow.sklearn.autolog(log_models=True)

    # Prepare the "Normalcy" data (Imbalanced, No SMOTE)
    X_train_normal = X_train_imb[y_train_imb == 0]
    
    # Initialize and Fit the Isolation Forest
    # Contamination represents the expected % of outliers
    iso_params = {"n_estimators": 100, "contamination": 0.01, "random_state": 42}
    iso_forest = IsolationForest(**iso_params)
    
    iso_forest.fit(X_train_normal)
    
    # Generate Anomaly Scores
    # Note: decision_function returns the signed distance to the cluster
    anomaly_scores = iso_forest.decision_function(X_test)
    
    # Log Custom Metrics (Optional but recommended for Azure)
    # We log the mean anomaly score to track drift over time
    mlflow.log_metric("mean_anomaly_score", anomaly_scores.mean())
    
    print("Isolation Forest training complete and logged to Azure ML.")

In [ ]:
# Create the analysis dataframe
layer2_df = pd.DataFrame({
    'Actual': y_test,
    'XGB_Prob': final_probs,
    'IF_Score': anomaly_scores
})

# Define the "Missed" set (XGBoost False Negatives at 0.95 threshold)
missed_by_xgb = layer2_df[(layer2_df['Actual'] == 1) & (layer2_df['XGB_Prob'] < 0.95)]

# Define the "Recovered" set (Those with a negative anomaly score)
# In Isolation Forest, scores < 0 are predicted anomalies
recovered_by_if = missed_by_xgb[missed_by_xgb['IF_Score'] < 0].sort_values(by='IF_Score')

# 4. Calculate Recovery Metrics for your README
total_missed = len(missed_by_xgb)
total_recovered = len(recovered_by_if)
recovery_rate = (total_recovered / total_missed) * 100 if total_missed > 0 else 0

print(f"--- Layer 2 Performance Summary ---")
print(f"Total Fraud Missed by Layer 1 (XGBoost): {total_missed}")
print(f"Total Fraud Recovered by Layer 2 (Isolation Forest): {total_recovered}")
print(f"Layer 2 Recovery Rate: {recovery_rate:.2f}%")
print("\n--- Identified 'Silent Frauds' (Negative Anomaly Scores) ---")
display(recovered_by_if)

In [ ]:
# Wrapper with Type Hints (solves UserWarning)
class FraudDetectionSystem(mlflow.pyfunc.PythonModel):
    def __init__(self, xgb_model, iso_forest, threshold=0.95):
        self.xgb_model = xgb_model
        self.iso_forest = iso_forest
        self.threshold = threshold

    def predict(self, context: Any, model_input: pd.DataFrame) -> pd.DataFrame:
        # Layer 1
        xgb_probs = self.xgb_model.predict_proba(model_input)[:, 1]
        # Layer 2
        iso_scores = self.iso_forest.decision_function(model_input)
        
        # Combined Decision
        final_preds = np.where((xgb_probs >= self.threshold) | (iso_scores < 0), 1, 0)
        
        return pd.DataFrame({
            "fraud_prediction": final_preds,
            "xgb_probability": xgb_probs,
            "anomaly_score": iso_scores
        })

# Setup paths and Signature
model_path = "final_hybrid_fraud_model"
if os.path.exists(model_path):
    shutil.rmtree(model_path)

# Instantiate
my_system = FraudDetectionSystem(best_xgb_model, iso_forest, 0.95)

# Infer signature (solves the schema/missing value warning)
# We use a sample of X_test to define the "contract"
signature = infer_signature(X_test.head(), my_system.predict(None, X_test.head()))

# Execution
with mlflow.start_run(run_name="Hybrid_System_Final_Fixed"):
    # Save the professional model structure locally
    mlflow.pyfunc.save_model(
        path=model_path,
        python_model=my_system,
        signature=signature
    )
    
    # Upload as artifact folder (Bypasses the 404 error)
    mlflow.log_artifacts(model_path, artifact_path="model")
    
    # Log key logic as parameters for the dashboard
    mlflow.log_param("layer1_threshold", 0.95)
    mlflow.log_param("layer2_type", "IsolationForest")
    
    print("Hybrid System logged as artifact successfully!")

In [ ]:
run_id = "bc1eb6c4-714f-43c4-90c0-c9870d2cd2c0"
model_name = "fraud-hybrid-champion"

# Create the Model object (The blueprint)
model_asset = Model(
    path=f"runs:/{run_id}/model",
    name=model_name,
    description="Hybrid XGBoost + Isolation Forest model",
    type=AssetTypes.MLFLOW_MODEL
)

# Register it
registered_model = ml_client.models.create_or_update(model_asset)
print(f"✅ Model registered! Version: {registered_model.version}")

In [ ]:
timestamp = datetime.datetime.now().strftime('%H%M')
endpoint_name = f"fraud-endpoint-{timestamp}"

print(f"Creating Endpoint: {endpoint_name}...")
endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="Fraud detection endpoint for testing",
    auth_mode="key"
)
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

# Deployment
print(f"Deploying model... (This may fail if total subscription quota is 0)")
deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=endpoint_name,
    model=f"azureml:fraud-hybrid-champion:1",
    instance_type="Standard_E2s_v3",
    instance_count=1,
)

try:
    ml_client.online_deployments.begin_create_or_update(deployment).result()
    
    # Assign Traffic
    endpoint.traffic = {"blue": 100}
    ml_client.online_endpoints.begin_create_or_update(endpoint).result()
    print(f"SUCCESS! Endpoint live at: {endpoint.scoring_uri}")

except Exception as e:
    print(f"Quota still full. Specific Error: {e}")
    print("\n💡 PORTFOLIO STRATEGY: Stop your Compute Instance (Notebook VM) in the 'Compute' tab.")
    print("Then, use the 'Deploy' button in the Azure ML Studio UI while the notebook is OFF.")

In [ ]:
# --- 1. CONFIGURATION ---
# Replace with the details found in the 'Consume' tab of your Azure ML Endpoint
ENDPOINT_URL = "https://fraud-serverless-1520.uksouth.inference.ml.azure.com/score"
API_KEY = "YOUR_SECURE_TOKEN" # In production, use os.getenv('AZURE_ML_KEY')

# --- 2. DATA PREPARATION ---
# We simulate selecting a 'Normal' and 'Fraud' case from the test set for verification
print("🧪 Preparing test samples from X_test...")
normal_pos = np.where(y_test == 0)[0][0]
fraud_pos = np.where(y_test == 1)[0][0]

# Extract feature rows using .iloc for position-based selection
samples = X_test.iloc[[normal_pos, fraud_pos]]
sample_list = samples.values.tolist()

# Construct the MLflow-compliant payload structure
payload = {
    "input_data": {
        "columns": X_test.columns.tolist(),
        "data": sample_list
    }
}

# --- 3. LIVE API INVOCATION ---
def invoke_fraud_endpoint(data_payload):
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}",
        "azureml-model-deployment": "blue" 
    }

    try:
        print(f"📡 Requesting inference from Azure Managed Endpoint...")
        response = requests.post(ENDPOINT_URL, json=data_payload, headers=headers)
        response.raise_for_status()
        
        results = response.json()
        
        # --- 4. DATA PRESENTATION ---
        # Create a clean comparison table for the recruiter
        output_df = pd.DataFrame({
            "Actual_Transaction": ["Normal", "Fraud"],
            "Predicted_Label": ["🚩 FRAUD" if p == 1 else "✅ NORMAL" for p in results['predictions']],
            "Anomaly_Score": results.get('scores', ["N/A"] * 2) # Isolation Forest Component
        })
        
        print("\n" + "="*45)
        print("🛡️  LIVE API VERIFICATION RESULTS  🛡️")
        print("="*45)
        print(output_df.to_string(index=False))
        print("="*45)
        
    except Exception as e:
        print(f"❌ API Call Failed: {e}")

# Run the test
invoke_fraud_endpoint(payload)

In [ ]:
# Delete the endpoint to save cost
print("Cleaning up: Deleting deployment to save credits...")
# We delete the deployment but can keep the endpoint name for your portfolio record
ml_client.online_deployments.begin_delete(name=deployment_name, endpoint_name=endpoint_name)

print("Deletion complete.")

In [ ]:
# Prepare data for testing
normal_pos = np.where(y_test == 0)[0][0]
fraud_pos = np.where(y_test == 1)[0][0]

# Extract the feature rows and convert to list for JSON serialization
samples = X_test.iloc[[normal_pos, fraud_pos]]
sample_list = samples.values.tolist()

# Define the MLflow-compliant payload
sample_payload = {
    "input_data": {
        "columns": X_test.columns.tolist(),
        "data": sample_list
    }
}

# Save payload to a local file for the SDK to upload
with open("guaranteed_test.json", "w") as f:
    json.dump(sample_payload, f)

# --- STEP 2: CLOUD INVOCATION WITH RETRY LOGIC ---
endpoint_name = 'fraud-serverless-1520'
max_retries = 3
retry_delay = 30  
res_dict = None

print(f"Sending comparison request (1 Normal, 1 Fraud) to: {endpoint_name}")

for i in range(max_retries):
    try:
        # Invoke the cloud endpoint
        response = ml_client.online_endpoints.invoke(
            endpoint_name=endpoint_name, 
            request_file="guaranteed_test.json"
        )
        res_dict = json.loads(response)
        print(f"Success: Model responded on attempt {i+1}!")
        break
    except Exception as e:
        if i < max_retries - 1:
            print(f"Attempt {i+1} failed (Likely Cold Start). Retrying in {retry_delay}s...")
            time.sleep(retry_delay)
        else:
            print("Error: Maximum retries reached. Connection failed.")
            raise e

# --- RESULTS PRESENTATION ---
if res_dict:
    # Construct a clean results table
    # Note: 'scores' or 'anomaly_scores' depends on your specific score.py output
    results_df = pd.DataFrame({
        "Actual_Label": ["Normal", "Fraud"],
        "Predicted_Class": ["Fraud" if p == 1 else "Normal" for p in res_dict.get('predictions', [])],
        "Anomaly_Score": res_dict.get('scores', res_dict.get('anomaly_scores', ["N/A"] * 2)),
        "Confidence_Prob": res_dict.get('probabilities', ["N/A"] * 2)
    })

    print("\n" + "="*40)
    print("HYBRID MODEL CLOUD VERIFICATION")
    print("="*40)
    print(results_df.to_string(index=False))
    print("="*40)
    print("Evidence of successful End-to-End Cloud Inference.")

In [ ]:
# Get the logs from the serverless deployment
# Note: For serverless, you might need to check the 'Monitoring' tab in the UI 
# if the SDK command below returns empty.
try:
    logs = ml_client.online_deployments.get_logs(
        name="blue", # Or the name you gave your deployment
        endpoint_name="fraud-serverless-1520", 
        lines=50
    )
    print(logs)
except Exception as e:
    print("Could not retrieve logs via SDK. Please check the 'Test' tab in Azure ML Studio for error details.")

In [ ]:
timestamp = datetime.datetime.now().strftime('%H%M%S')
endpoint_name = f"fraud-final-{timestamp}"

print(f"Deploying to FRESH endpoint: {endpoint_name}")

# Create the Endpoint
endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="Fresh deployment after fix",
    auth_mode="key"
)
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

# Deploy the Blue version
blue_deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=endpoint_name,
    model=f"azureml:fraud-hybrid-champion:1",
    instance_type="Standard_DS1_v2",
    instance_count=1
)

print("Provisioning compute... (this is the 8-10 minute part)")
ml_client.online_deployments.begin_create_or_update(blue_deployment).result()

# 5. Route Traffic
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

print(f"DEPLOYED SUCCESSFULLY!")
print(f"URL: {ml_client.online_endpoints.get(endpoint_name).scoring_uri}")

In [ ]:
endpoint = ml_client.online_endpoints.get(name="fraud-final-145607")

print(f"Endpoint State: {endpoint.provisioning_state}")
if endpoint.scoring_uri:
    print(f"✅ Live URL: {endpoint.scoring_uri}")
else:
    print("⏳ Still provisioning...")

In [ ]:
# Get the data as a list of lists
sample_data = X_test.head(10).values.tolist() 

# Define the payload
sample_payload = {
    "input_data": {
        # Ensure these column names match your training X_test exactly
        "columns": X_test.columns.tolist(), 
        "data": sample_data 
    }
}

# Save to JSON
with open("test_request.json", "w") as f:
    json.dump(sample_payload, f)

# 4. Invoke the endpoint
response = ml_client.online_endpoints.invoke(
    endpoint_name='fraud-final-145607',
    request_file="test_request.json"
)

print(f"Cloud Response: {response}")

In [ ]:
# Replace 'blue' if you named your deployment something else
logs = ml_client.online_deployments.get_logs(
    name="blue", 
    endpoint_name="fraud-final-145607", 
    lines=50
)
print(logs)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import PrecisionRecallDisplay, average_precision_score, classification_report, confusion_matrix
import pandas as pd
import seaborn as sns

# Extract the best estimators
best_rf = random_search_rf.best_estimator_
best_xgb = random_search_xgb.best_estimator_

# Generate Predictions
rf_probs = best_rf.predict_proba(X_test)[:, 1]
xgb_probs = best_xgb.predict_proba(X_test)[:, 1]

# Calculate Final AUPRC Scores
rf_auprc = average_precision_score(y_test, rf_probs)
xgb_auprc = average_precision_score(y_test, xgb_probs)

# Setup the Figure
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# --- PANEL 1: Precision-Recall Curve ---
ax1 = axes[0]
PrecisionRecallDisplay.from_estimator(best_rf, X_test, y_test, ax=ax1, name="Tuned RF")
PrecisionRecallDisplay.from_estimator(best_xgb, X_test, y_test, ax=ax1, name="Tuned XGB")
ax1.set_title("PR Curve: RF vs XGBoost")
ax1.grid(True, alpha=0.3)

# --- PANEL 2: Confusion Matrix (RF) ---
y_pred_rf = best_rf.predict(X_test)
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', ax=axes[1], cbar=False)
axes[1].set_title("RF: Actual vs Predicted")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

# --- PANEL 3: Confusion Matrix (XGBoost) ---
y_pred_xgb = best_xgb.predict(X_test)
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Greens', ax=axes[2], cbar=False)
axes[2].set_title("XGBoost: Actual vs Predicted")
axes[2].set_xlabel("Predicted")

plt.tight_layout()
plt.show()